In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

In [ ]:
from collections import Counter

import jlens
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import transformers
from datasets import load_from_disk
from tqdm.auto import tqdm

from experiments.jlens_readout_sanity.constants import (
    LENS_PATH,
    MODEL_NAME,
    MODEL_PATH,
)
from jlens_reasoning.benchmarks.flenqa.dataset import (
    normalize_rows,
    prepare_prompts,
)
from jlens_reasoning.benchmarks.flenqa.lens import (
    ApplyLensRunner,
    LensRunners,
)
from jlens_reasoning.benchmarks.flenqa.runner import (
    RunConfig,
    run_benchmark,
)
from jlens_reasoning.inference import InferenceConfig, generate_chat

dataset = load_from_disk(context.datasets_dir / "flenqa")
raw_rows = dataset["eval"] if hasattr(dataset, "keys") else dataset
rows = normalize_rows(raw_rows, full=True)
causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
causal_lm.eval()
model = jlens.from_hf(causal_lm, tokenizer)
lens = jlens.JacobianLens.from_pretrained(LENS_PATH)
runners = LensRunners(
    ApplyLensRunner(lens, model, True), ApplyLensRunner(lens, model, False)
)

In [ ]:
len(rows)

In [ ]:
summary = run_benchmark(
    rows,
    output_dir=context.runs_dir / "flenqa-full-run",
    tokenizer=tokenizer,
    runners=runners,
    config=RunConfig(
        top_k=250,
        expected_source_rows=12_000,
    ),
)
summary

In [ ]:
MODEL_OUTPUT_PATH = context.runs_dir / "flenqa-full-run" / "model_outputs.parquet"
LENGTHS = (250, 500, 1000, 2000, 3000)
EXPECTED_UNIQUE_COUNTS = {
    250: 300,
    500: 2_368,
    1000: 2_394,
    2000: 2_400,
    3000: 2_400,
}
INFERENCE_CONFIG = InferenceConfig.direct(max_input_tokens=4096)
MODEL_OUTPUT_SCHEMA = pa.schema(
    [
        pa.field("prompt_id", pa.string(), nullable=False),
        pa.field("problem_id", pa.int32(), nullable=False),
        pa.field("task", pa.string(), nullable=False),
        pa.field("label", pa.bool_(), nullable=False),
        pa.field("text", pa.string(), nullable=False),
        pa.field("ctx_size", pa.int32(), nullable=False),
        pa.field("n_input_tokens", pa.int32(), nullable=False),
        pa.field("paper_weight", pa.int16(), nullable=False),
        pa.field("model_name", pa.string(), nullable=False),
        pa.field("code_revision", pa.string(), nullable=False),
        pa.field("inference_mode", pa.string(), nullable=False),
        pa.field("max_new_tokens", pa.int32(), nullable=False),
        pa.field("do_sample", pa.bool_(), nullable=False),
        pa.field("temperature", pa.float32()),
        pa.field("top_p", pa.float32()),
        pa.field("top_k", pa.int32()),
        pa.field("min_p", pa.float32()),
        pa.field("generated_token_ids", pa.list_(pa.int32()), nullable=False),
        pa.field("generated_token_pieces", pa.list_(pa.string()), nullable=False),
        pa.field("generated_text", pa.string(), nullable=False),
        pa.field("reasoning_text", pa.string()),
        pa.field("answer_text", pa.string()),
        pa.field("reasoning_status", pa.string(), nullable=False),
        pa.field("generation_status", pa.string(), nullable=False),
        pa.field("finish_reason", pa.string()),
    ]
)

prompts = prepare_prompts(rows)
records = []
for prompt in tqdm(prompts, desc="FLenQA model outputs", unit="prompt"):
    ctx_sizes = {item.ctx_size for item in prompt.provenance}
    if len(ctx_sizes) != 1:
        raise ValueError(f"Prompt spans nominal lengths: {sorted(ctx_sizes)}")
    inference = generate_chat(
        causal_lm,
        tokenizer,
        prompt.text,
        config=INFERENCE_CONFIG,
    )
    records.append(
        {
            "prompt_id": prompt.prompt_id,
            "problem_id": prompt.problem_id,
            "task": prompt.task,
            "label": prompt.label,
            "text": prompt.text,
            "ctx_size": ctx_sizes.pop(),
            "n_input_tokens": inference.input_token_count,
            "paper_weight": sum(
                item.dispersion == "random" for item in prompt.provenance
            ),
            "model_name": MODEL_NAME,
            "code_revision": PROJECT_COMMIT,
            "inference_mode": inference.config.mode.value,
            "max_new_tokens": inference.config.max_new_tokens,
            "do_sample": inference.config.do_sample,
            "temperature": inference.config.temperature,
            "top_p": inference.config.top_p,
            "top_k": inference.config.top_k,
            "min_p": inference.config.min_p,
            "generated_token_ids": list(inference.output.token_ids),
            "generated_token_pieces": list(inference.output.token_pieces),
            "generated_text": inference.raw_text,
            "reasoning_text": inference.reasoning_text,
            "answer_text": inference.answer_text,
            "reasoning_status": inference.reasoning_status.value,
            "generation_status": inference.output.generation_status.value,
            "finish_reason": inference.output.finish_reason,
        }
    )

assert len(records) == 9_862
actual_counts = Counter(record["ctx_size"] for record in records)
assert dict(actual_counts) == EXPECTED_UNIQUE_COUNTS
paper_counts = Counter()
for record in records:
    paper_counts[record["ctx_size"]] += record["paper_weight"]
assert dict(paper_counts) == {length: 600 for length in LENGTHS}

model_outputs = pa.Table.from_pylist(records, schema=MODEL_OUTPUT_SCHEMA)
assert model_outputs.num_rows == 9_862
pq.write_table(model_outputs, MODEL_OUTPUT_PATH, compression="zstd")
MODEL_OUTPUT_PATH